# Python Basics Exercises

## 1) Numeric Variable Types — Exercises
- Learning goals: integers vs floats, integer division, modulo, rounding, conversions, boolean math, overflow (arbitrary precision), simple aggregates

### Warm-ups

1) Type explorer
- Write num_type(x) → return the exact type name ("int", "float", "bool").
- def num_type(x):
- assert num_type(3) == "int"
- assert num_type(2.0) == "float"
- assert num_type(True) == "bool"

In [26]:
def num_type(x):
    # Check the type of x and return the corresponding string.
    if isinstance(x, int) and not isinstance(x, bool):
        # We must exclude 'isinstance(x, bool)' because 'bool' is a
        # subclass of 'int' and True/False can be treated as 1/0.
        return "int"
    elif isinstance(x, float):
        return "float"
    elif isinstance(x, bool):
        return "bool"
    # Optional: Fallback for other types, if needed
    return "unknown"

# Test-Assertions (Überprüfungen)
assert num_type(3) == "int"
assert num_type(2.0) == "float"
assert num_type(True) == "bool"
assert num_type(complex) == "unknown"
print("Task 1 successfully completed!")

Task 1 successfully completed!


2) Safe division
- Write safe_div(a, b, default=None) that returns a/b as float; if b==0 return default.
- def safe_div(a, b, default=None):
-     ...
- assert safe_div(6, 3) == 2.0
- assert safe_div(1, 0, default=float('inf')) == float('inf')


In [27]:
def safe_div(a, b, default=None):
    # Check if the denominator (b) is zero
    if b == 0:
        return default
    else:
        # Perform the standard division (/) which always returns a float.
        return a / b

# Test-Assertions
assert safe_div(6, 3) == 2.0
# float('inf') stands for "infinity" (infinity)
assert safe_div(1, 0, default=float('inf')) == float('inf')
print("Task 2 successfully completed!")

Task 2 successfully completed!


3) Integer vs float division
- Write divisions(a, b) → (a // b, a / b, a % b); handle negatives correctly.
- def divisions(a, b):
-     ...
- assert divisions(7, 3) == (2, 7/3, 1)
- assert divisions(-7, 3)[0] == -3   # floor division

In [28]:
def divisions(a, b):
    # a // b: Floor Division 
    # a / b: Float Division 
    # a % b: Modulo 
    return (a // b, a / b, a % b)

# Test-Assertions
# 7 / 3 equals 2 rest 1 -> (2, 2.333..., 1)
assert divisions(7, 3) == (2, 7/3, 1)

# -7 / 3 ist -3 Rest 2. (Da -3 * 3 + 2 = -7)
# The Floor Division always rounds to negative infinity!
assert divisions(-7, 3)[0] == -3
print("Task 3 successfully completed!")

Task 3 successfully completed!


### Core

4) Rounding modes
- Write bankers_round(x, ndigits=0) using round() (banker’s rounding) and classic_round(x, ndigits=0) emulating away-from-zero rounding.
- def bankers_round(x, ndigits=0):
-     ...
- def classic_round(x, ndigits=0):
-     ...
- assert bankers_round(2.5) == 2
- assert classic_round(2.5) == 3

In [46]:
import math # Import the math module for floor/ceil operations (though not strictly needed here)

## 1. Banker's Rounding (Uses standard Python round())
def bankers_round(x, ndigits=0):
    # Rounds a number using the standard Python 'round()' function (Banker's Rounding).
    # This rounds to the nearest even number when a value is exactly halfway (e.g., X.5).
    
    # Simply use the built-in round function
    return round(x, ndigits)

## 2. Classic Rounding (Away-from-zero)
def classic_round(x, ndigits=0):
    
    # Emulates Classic Rounding (Half Up or Away-from-Zero) by adding/subtracting 0.5 
    # after shifting the decimal point, ensuring that X.5 always rounds up (or down, 
    # away from zero, for negatives).
    
    if ndigits < 0:
        # Handle negative ndigits (rounding to the left of the decimal)
        factor = 10**ndigits
        
    # 1. Shift the number so the rounding position is the ones place
    factor = 10**ndigits
    shifted_x = x * factor

    # 2. Add 0.5 (for positive numbers) or subtract 0.5 (for negative numbers)
    #    This ensures that .5 cases are pushed over the int() boundary, 
    #    making the rounding "away from zero".
    if shifted_x >= 0:
        # Add 0.5 for positive numbers
        rounded_shifted = int(shifted_x + 0.5)
    else:
        # Subtract 0.5 for negative numbers (e.g. -2.5 - 0.5 = -3.0 -> int(-3.0) = -3)
        rounded_shifted = int(shifted_x - 0.5)

    # 3. Shift the number back
    return rounded_shifted / factor

# Test Assertions
assert bankers_round(2.5) == 2
assert classic_round(2.5) == 3
assert classic_round(-2.5) == -3
assert classic_round(2.2) == 2  # Not exactly .5, so standard rounding applies
assert classic_round(2.8) == 3  # Not exactly .5, so standard rounding applies
assert classic_round(1.235, 2) == 1.24 # Example with 2 ndigits
assert classic_round(1234, -2) == 1200 # Example with negative ndigits
print("Exercise 4 successfully completed and re-verified!")

Exercise 4 successfully completed and re-verified!


In [47]:
# Special case ndigits=0 (rounding to the nearest integer).
def classic_round_only_integers(x):
    # This simplified version only works when rounding to the nearest integer!
    if x >= 0:
        return int(x + 0.5)
    else:
        return int(x - 0.5)

assert classic_round_only_integers(2.5) == 3 # Correct
assert classic_round_only_integers(1.235) == 1 # WRONG (should be 1.24 if ndigits=2)

5) Parse numbers
- Write parse_int(s, base=10, default=None) that trims whitespace, supports underscores ("1_000"), and returns default on ValueError.
- def parse_int(s, base=10, default=None):
-     ...
- assert parse_int(" 1_024 ") == 1024 
- assert parse_int("FF", base=16) == 255
- assert parse_int("oops", default=-1) == -1

In [49]:
def parse_int(s, base=10, default=None):
    # 1. Try to prepare and convert the string.
    try:
        # Remove trailing spaces and underscores from the string.
        cleaned_s = s.strip().replace("_", "")
        # 2. Convert the cleaned string to an integer.
        return int(cleaned_s, base=base)
    # 3. Catch the error if the conversion fails.
    except ValueError:
        return default

# Test-Assertions
assert parse_int(" 1_024 ") == 1024
assert parse_int("FF", base=16) == 255
assert parse_int("oops", default=-1) == -1
print("Exercise 5 successfully completed and verified!")

Exercise 5 successfully completed and verified!


6) Boolean arithmetic
- Write true_ratio(seq) that returns the fraction of truthy values in seq.
- def true_ratio(seq):
-     ...
- assert abs(true_ratio([True, False, 1, 0, "", "x"]) - 0.5) < 1e-9

In [50]:
def true_ratio(seq):
    # Returns the ratio (fraction) of truthy values in the sequence 'seq'.
    # The built-in sum() function treats True as 1 and False as 0, 
    # allowing us to count the number of truthy items easily.
        
    # 1. Handle the edge case where the sequence is empty to prevent ZeroDivisionError.
    if not seq:
        return 0.0

    # 2. Count the number of truthy values:
    #    sum(seq) adds up all elements. Truthy elements (like 1, "x", True) are counted as 1.
    num_truthy = sum(seq)

    # 3. Calculate the total number of elements.
    total_elements = len(seq)

    # 4. Calculate and return the ratio. We ensure float division by returning the float result.
    return num_truthy / total_elements

# Test Assertions
# [True (1), False (0), 1 (1), 0 (0), "" (0), "x" (1)] -> Sum = 3. Length = 6. Ratio = 3/6 = 0.5
assert abs(true_ratio([True, False, 1, 0, "", "x"]) - 0.5) < 1e-9
print("Exercise 6 successfully completed!")

TypeError: unsupported operand type(s) for +: 'int' and 'str'

In [52]:
def true_ratio(seq):
    # Returns the ratio (fraction) of truthy values in the sequence 'seq'.
    # The generator expression (bool(x) for x in seq) ensures that every element 
    # is first evaluated for its truthiness (True/False) before summing them up 
    # as 1s and 0s. This prevents the TypeError with mixed types like strings.
        
    # 1. Handle the edge case where the sequence is empty.
    if not seq:
        return 0.0

    # 2. Correct way to count the number of truthy values:
    #    - bool(x) converts each element into True or False (truthiness).
    #    - sum() then treats True as 1 and False as 0.
    #    This avoids attempting to add non-numeric types like strings.
    num_truthy = sum(bool(x) for x in seq)

    # 3. Calculate the total number of elements.
    total_elements = len(seq)

    # 4. Calculate and return the ratio.
    return num_truthy / total_elements

# Test Assertions
# The list contains numbers and strings, which the corrected function now handles.
assert abs(true_ratio([True, False, 1, 0, "", "x"]) - 0.5) < 1e-9
print("Exercise 6 successfully completed and verified!")

Exercise 6 successfully completed and verified!


7) Min/Max & aggregates
- Write stats(nums) → dict with count, total, mean, minimum, maximum. Empty list → return zeros/None appropriately.
- def stats(nums):
-     ...
- assert stats([1,2,3])["mean"] == 2

In [53]:
def stats(nums):
    # Calculates basic aggregate statistics (count, total, mean, min, max) 
    # for a list of numbers.
    # Returns a dictionary. Handles an empty list gracefully by returning 
    # appropriate zero or None values.
        
    # 1. Handle the edge case: Empty list
    if not nums:
        return {
            "count": 0,
            "total": 0,
            "mean": None,    # Mean is undefined for an empty set
            "minimum": None, # Minimum is undefined for an empty set
            "maximum": None, # Maximum is undefined for an empty set
        }

    # 2. Calculate simple aggregates using built-in functions
    count = len(nums)
    total = sum(nums)
    minimum = min(nums)
    maximum = max(nums)
    
    # 3. Calculate the mean
    # Since we already checked 'if not nums', we know count > 0, so no ZeroDivisionError
    mean = total / count

    # 4. Return the results as a dictionary
    return {
        "count": count,
        "total": total,
        "mean": mean,
        "minimum": minimum,
        "maximum": maximum,
    }

# Test Assertions
# Test case 1: Normal list
result = stats([1, 2, 3])
assert result["mean"] == 2
assert result["minimum"] == 1
assert result["count"] == 3

# Test case 2: Empty list
result_empty = stats([])
assert result_empty["count"] == 0
assert result_empty["mean"] == None
assert result_empty["minimum"] == None

print("Exercise 7 successfully completed!")

Exercise 7 successfully completed!


8) Bucket by step
- Write bucket(x, step) → largest multiple of step ≤ x (works with negatives).
- def bucket(x, step):
-     ...
- assert bucket(37, 10) == 30
- assert bucket(-3, 5) == -5

In [54]:
def bucket(x, step):
    # Finds the largest multiple of 'step' that is less than or equal to 'x'.
    # This is achieved by using Python's floor division (//).
        
    # 1. Handle the edge case where step is zero (though often not explicitly required, it's safer)
    if step == 0:
        # Cannot determine a multiple of zero, which results in division by zero.
        # Returning x is a common approach or raising an error. We'll return x as a safe fallback.
        return x 
        # Alternatively: raise ValueError("Step cannot be zero")

    # 2. Perform floor division to count how many times 'step' fits into 'x' (always rounding down).
    # This works correctly for both positive and negative numbers.
    count = x // step
    
    # 3. Multiply the count back by 'step' to get the largest multiple of 'step' <= x.
    return count * step

# Test Assertions
assert bucket(37, 10) == 30
assert bucket(-3, 5) == -5
assert bucket(10, 10) == 10 # Test case: exact multiple
assert bucket(-10, 3) == -12 # Test case: negative numbers not exactly divisible
print("Exercise 8 successfully completed!")

Exercise 8 successfully completed!


9) Time math (seconds → h:m:s)
- Write to_hms(seconds) → "HH:MM:SS" with zero-padding.
- def to_hms(seconds):
-     ...
- assert to_hms(3661) == "01:01:01"

In [55]:
def to_hms(seconds):
    # Converts a total number of seconds into the "HH:MM:SS" time format 
    # with zero-padding.
        
    # 1. Calculate Hours
    # Total hours: Use floor division (//) to get the whole number of hours.
    hours = seconds // 3600
    
    # Calculate the remaining seconds after removing the hours.
    remaining_seconds = seconds % 3600

    # 2. Calculate Minutes
    # Minutes: Use floor division on the remainder to get the whole minutes.
    minutes = remaining_seconds // 60

    # 3. Calculate Final Seconds
    # Final seconds: Use modulo on the remainder to get the remaining seconds (< 60).
    secs = remaining_seconds % 60

    # 4. Format the final string
    # Use an f-string with the formatting specifier ':02d' 
    # to ensure each number is printed as a decimal integer (d) 
    # with a leading zero (0) and a width of two characters (2).
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

# Test Assertions
# 3661 seconds = 1 hour + 1 minute + 1 second
assert to_hms(3661) == "01:01:01"
# 125 seconds = 0 hours + 2 minutes + 5 seconds
assert to_hms(125) == "00:02:05"
print("Exercise 9 successfully completed!")

Exercise 9 successfully completed!


### Challenge

10) Running totals with precise rounding
- Write running_totals(amounts, ndigits=2) that returns cumulative sums rounded at each step (like financial ledgers) using banker’s rounding. Example: [10.005, 0.005] → [10.01, 10.02].
- def running_totals(amounts, ndigits=2):
-     ...
- assert running_totals([10.005, 0.005], 2) == [10.01, 10.02]

In [60]:
def running_totals(amounts, ndigits=2):
    # Calculates the cumulative sum (running total) of a list of amounts, 
    # applying Banker's Rounding at each step to the specified number of digits.
    # This mimics the precision required in financial ledgers.
        
    # Initialize the list to store the results
    running_totals_list = []
    
    # Initialize the running total (the cumulative state)
    current_total = 0.0

    # Iterate through each amount in the input list
    for amount in amounts:
        
        # 1. Update the total with the new amount
        current_total += amount # same as current_total = current_total + amount

        # 2. Apply Banker's Rounding (Python's built-in round()) immediately.
        #    This is key: the total is rounded BEFORE the next amount is added.
        #    We reassign the rounded value back to current_total.
        current_total = round(current_total, ndigits)

        # 3. Append the precisely rounded result to the output list
        running_totals_list.append(current_total)

    return running_totals_list

# Test Assertions
# Step 1: 10.005 -> round(10.005, 2) = 10.01 (Rounds to nearest even digit, which is 10.00 + 0.01)
# Step 2: 10.01 (rounded total) + 0.005 -> 10.015
# Step 3: round(10.015, 2) = 10.02 (Rounds to nearest even digit, which is 10.02)
assert running_totals([10.005, 0.005], 2) == [10.01, 10.02]

# Another test: demonstrating standard rounding 
print(running_totals([1.113, 0.005, 0.005], 2) )#== [1.11, 1.12, 1.12])
print("Exercise 10 (Challenge) successfully completed!")

[1.11, 1.11, 1.11]
Exercise 10 (Challenge) successfully completed!


## 2) Strings — Exercises

### Warm-ups

1) Middle char(s)
- middle(s) → If odd length return the middle char, else return the two middle chars.
- def middle(s):
-     ...
- assert middle("abc") == "b"
- assert middle("abba") == "bb"

In [61]:
def middle(s):
    # Returns the middle character (odd length) or two middle characters (even length) 
    # of a string 's'.
    
    length = len(s)
    
    # Calculate the index where the middle starts
    # For odd length (e.g., 5), middle_index = 5 // 2 = 2. Slicing starts at 2.
    # For even length (e.g., 4), middle_index = 4 // 2 = 2. Slicing starts at 1 (2-1).
    middle_index = length // 2
    
    if length % 2 == 1:
        # Odd length: return the single middle character. Index is length // 2.
        return s[middle_index]
    else:
        # Even length: return the two middle characters. Indices are [middle_index - 1] and [middle_index].
        # Slicing from index - 1 up to index + 1 (exclusive) covers the two middle elements.
        return s[middle_index - 1 : middle_index + 1]

assert middle("abc") == "b"
assert middle("abba") == "bb"
# assert middle("abcdefg") == "d" # Test for longer odd string
# assert middle("abcdef") == "cd" # Test for longer even string

2) Title-case safely
- safe_title(s) → title case but do not lowercase words fully in ALL CAPS acronyms (e.g., "learn SQL now" → "Learn SQL Now"; "use GPU" → "Use GPU").
- def safe_title(s):
-    ...

In [65]:
def safe_title(s):
    # Converts a string to title case but preserves words that are entirely 
    # in uppercase (acronyms) to prevent lowercasing them.
        
    # 1. Split the string into a list of words.
    words = s.split()
    result_words = []
    
    for word in words:
        # 2. Check if the word is an acronym (all characters are uppercase)
        if word.isupper():
            # If it is an acronym, append it as is.
            result_words.append(word)
        else:
            # If it is not an acronym, apply standard title case.
            # .title() makes the first letter upper and the rest lower.
            result_words.append(word.title())
            
    # 3. Join the processed words back together with a single space.
    return " ".join(result_words)

# Test Assertions
assert safe_title("learn SQL now") == "Learn SQL Now"
assert safe_title("use GPU") == "Use GPU"
assert safe_title("check the API documentation") == "Check The API Documentation"
print(safe_title("check the API documentation"))

Check The API Documentation


3) Reverse words
- reverse_words(s) → reverse word order but keep internal characters.
- def reverse_words(s):
-     ...
- assert reverse_words("one two  three") == "three two  one"

In [66]:
def reverse_words(s):
    # Reverses the order of words in a string while preserving the exact original 
    # spacing structure by splitting and joining on a single space character.
        
    # 1. Split the string by a single space (' '). 
    # This ensures that runs of multiple spaces result in empty strings ('') in the list.
    words_and_spaces = s.split(' ')
    
    # 2. Reverse the list of elements (words and empty strings/spaces).
    words_and_spaces.reverse()
    
    # 3. Join the elements back using a single space as the delimiter.
    # The empty strings are now positioned to recreate the original spacing, 
    # but the words are in reverse order.
    return ' '.join(words_and_spaces)

# Test Assertions
assert reverse_words("one two  three") == "three  two one"
assert reverse_words(" leading space ") == " space leading "
assert reverse_words("a") == "a"

### Core

4) Normalize spaces
- squash_spaces(s) → collapse multiple spaces/tabs to a single space; strip ends.
- def squash_spaces(s):
-     ...
- assert squash_spaces("  a\t b   c ") == "a b c"

In [74]:
def squash_spaces(s):
    # Normalizes whitespace in a string: removes leading/trailing whitespace 
    # and collapses all internal sequences of spaces/tabs/newlines into a single space.
        
    # 1. Use split() without arguments. 
    # This automatically splits by any sequence of whitespace (tabs, multiple spaces, etc.) 
    # and implicitly handles stripping the ends, resulting in a clean list of words.
    words = s.split()
    
    # 2. Join the clean list of words back using a single space (' ').
    # This ensures perfect normalization: exactly one space between words.
    return " ".join(words)

# Test Assertions
# The input contains leading/trailing spaces, multiple spaces, and a tab (\t).
assert squash_spaces("  a\t b   c ") == "a b c"
assert squash_spaces("\tfirst  \n second\t") == "first second"

5) CSV line → list (no quotes)
- split_csv(s) → split by commas, trim whitespace around fields; ignore empty trailing field if line ends with comma.
- def split_csv(s):
-     ...
- assert split_csv(" a, b ,c,") == ["a","b","c"]

In [75]:
def split_csv(s):
    # Splits a comma-separated line, trimming whitespace from each field. 
    # Ignores an empty trailing field if the line ends with a comma 
    # (a common requirement for simple CSV parsing).
        
    # 1. Split the string by the comma delimiter.
    fields = s.split(',')
    
    # 2. Use a list comprehension to strip whitespace from every field.
    stripped_fields = [field.strip() for field in fields]
    
    # 3. Handle the trailing comma edge case.
    # The split() method always produces an empty string at the end if 
    # the delimiter is the last character (e.g., "a,b," -> ["a", "b", ""]).
    # We remove this trailing empty field ONLY if it was caused by a trailing comma.
    
    # Check if the original string ends with a comma AND
    # if the list is not empty AND the last field is an empty string
    if s.endswith(',') and stripped_fields and stripped_fields[-1] == "":
        # Exclude the last element using slicing [:-1]
        return stripped_fields[:-1]
    
    # Handle the case where the input is an empty string, 
    # which correctly splits to [""] and should be returned as is.
    return stripped_fields

# Test Assertions
assert split_csv(" a, b ,c,") == ["a","b","c"] # Trailing comma removed
assert split_csv(" x,y ") == ["x", "y"]       # Trimming whitespace
assert split_csv("a,b,,c") == ["a", "b", "", "c"] # Internal empty field is kept
assert split_csv("") == [""]                   # Empty input is not modified

6) Mask secrets
- mask_email(s) → keep first char of user and domain, mask the rest with *, keep TLD.
- "alice@example.com" → "a****@e******.com"
- def mask_email(s):

In [87]:
def mask_email(s):
    # Masks the user and domain parts of an email, keeping only the first 
    # character of each and the Top-Level Domain (TLD).
        
    # 1. Split user from domain using the '@' symbol
    try:
        user, full_domain = s.split('@')
    except ValueError:
        # If the string is not a valid email format (e.g., no '@'), return it as is.
        return s 

    # 2. Split domain from TLD. 
    # rsplit('.', 1) splits only on the rightmost (last) occurrence of the delimiter.
    # This correctly handles subdomains like 'sub.example.com'.
    try:
        domain_name, tld = full_domain.rsplit('.', 1)
    except ValueError:
        # If there is no dot (e.g., user@localhost), return the masked user and the domain as is.
        return f"{user[0]}{'*' * (len(user) - 1)}@{full_domain}"

    # 3. Mask the user part
    # Keep the first char [0], mask the rest (length - 1)
    user_mask_length = len(user) - 1
    masked_user = user[0] + "*" * user_mask_length
    
    # 4. Mask the domain name part (excluding TLD)
    # Keep the first char [0], mask the rest (length - 1)
    domain_mask_length = len(domain_name) - 1
    masked_domain_name = domain_name[0] + "*" * domain_mask_length
    
    # 5. Reconstruct the masked email
    return f"{masked_user}@{masked_domain_name}.{tld}"

# Test Assertions
assert mask_email("alice@example.com") == "a****@e******.com"
assert mask_email("max.pain@example.com") == "m*******@e******.com"
assert mask_email("j@longdomain.co.uk") == "j@l************.uk" # TLD is .co.uk, domain is longdomain
assert mask_email("name@test.net") == "n***@t***.net"
print(mask_email("j@longdomain.co.uk"))
print(mask_email("john@localhost"))


j@l************.uk
j***@localhost


7) Find all indexes
- find_all(s, sub) → list of start indices where sub occurs (including overlaps).
- def find_all(s, sub):
-     ...
- assert find_all("aaaa", "aa") == [0,1,2]

In [89]:
def find_all(s, sub):
    # Finds all starting indices of a substring 'sub' within string 's', 
    # including overlapping occurrences, using an iterative approach.
    
    indices = []
    start_index = 0
    
    # Check for trivial cases where the search or the substring is empty
    if not sub or not s:
        return []

    # Loop indefinitely until the substring is no longer found
    while True:
        # 1. Search for the next occurrence of 'sub' starting from 'start_index'
        index = s.find(sub, start_index)
        
        # 2. Check termination condition
        if index == -1:
            # If find() returns -1, the substring was not found, so we exit the loop.
            break
        
        # 3. Record the found index
        indices.append(index)
        
        # 4. Crucial step for overlaps: Advance the search position by 1.
        #    We only move one position forward (index + 1) to allow the next search 
        #    to begin at the second character of the current match.
        #    Example: If "aa" is found at [0], the next search starts at [1] to find the overlap.
        start_index = index + 1
        
    return indices

# Test Assertions
assert find_all("aaaa", "aa") == [0, 1, 2]     # Overlapping check
assert find_all("banana", "ana") == [1, 3]     # Standard check
assert find_all("ababab", "aba") == [0, 2] # More complex overlap

8) Anagram check
- is_anagram(a, b) ignoring spaces, case, punctuation.Anagram
- def is_anagram(a, b):
-     ...
- assert is_anagram("Listen", "Silent")

In [ ]:
import string

def is_anagram(a, b):
    # Checks if two strings 'a' and 'b' are anagrams. 
    # It ignores case, spaces, and punctuation during the comparison.
        
    def normalize_string(s):
        # Helper function to clean a string for comparison.
        
        # 1. Convert to lowercase
        s = s.lower()
        
        # 2. Remove all spaces and punctuation.
        # We use a string translation table for efficient removal:
        # str.maketrans creates a translation map where all characters 
        # in the second argument are mapped to None (meaning deletion).
        removal_map = str.maketrans('', '', string.punctuation + string.whitespace)
        
        # Apply the mapping
        s = s.translate(removal_map)
        
        return s

    # Normalize both input strings
    norm_a = normalize_string(a)
    norm_b = normalize_string(b)

    # 3. Anagram check: Compare the sorted lists of characters.
    # If the sorted lists are equal, the original strings were anagrams.
    return sorted(norm_a) == sorted(norm_b)

# Test Assertions
assert is_anagram("Listen", "Silent")
assert is_anagram("dormitory", "dirty room") # Ignores spaces
assert is_anagram("A gentleman!", "Elegant man") # Ignores spaces and punctuation
assert not is_anagram("Hello", "World")

In [92]:
def is_anagram_basic(a, b):
    """
    Checks if two strings are anagrams, ignoring case, spaces, and punctuation.
    This version avoids advanced methods like str.maketrans/translate 
    by using iteration and filtering (isalnum).
    """
    
    def normalize_basic(s):
        """
        Helper function to clean a string using basic iteration and isalnum().
        """
        cleaned_chars = []
        
        # 1. Convert the entire string to lowercase first
        s = s.lower()
        
        # 2. Iterate over every character in the string
        for char in s:
            # 3. Check if the character is alphanumeric (a-z or 0-9).
            #    This excludes spaces, punctuation, and other symbols.
            if char.isalnum():
                cleaned_chars.append(char)
                
        # 4. Join the list of valid characters back into a clean string
        return "".join(cleaned_chars)

    # Normalize both input strings
    norm_a = normalize_basic(a)
    norm_b = normalize_basic(b)

    # 5. Anagram check: Compare the sorted lists of characters.
    # If the sorted lists are equal, the original strings were anagrams.
    return sorted(norm_a) == sorted(norm_b)

# Test Assertions
assert is_anagram_basic("Listen", "Silent")
assert is_anagram_basic("dormitory", "dirty room")
assert is_anagram_basic("A gentleman!", "Elegant man")

9) Format table row
- fmt_row(values, widths) → left-align strings to fixed widths, joined by " | ".
- def fmt_row(values, widths):
-     ...
- assert fmt_row(["a","bb"], [3,4]) == "a   | bb  "

In [93]:
def fmt_row(values, widths):
    """
    Formats a list of values into a table row, left-aligned to specified widths, 
    and joined by ' | '. Uses f-string nested formatting for alignment.
    """
    formatted_parts = []
    
    # 1. Use zip() to iterate over the value and its corresponding width simultaneously
    for value, width in zip(values, widths):
        
        # Ensure the value is a string (important if values contains numbers)
        s_value = str(value)
        
        # 2. Apply nested f-string formatting for left alignment (<) and fixed width
        # {s_value:<{width}} means: take s_value, left-align (<) it, and make the 
        # total field width equal to the variable 'width'.
        formatted_part = f"{s_value:<{width}}"
        
        formatted_parts.append(formatted_part)
        
    # 3. Join the formatted parts using the specified separator (" | ")
    return " | ".join(formatted_parts)

# Test Assertions
# "a" (length 1) in width 3 -> "a  "
# "bb" (length 2) in width 4 -> "bb  "
assert fmt_row(["a","bb"], [3,4]) == "a   | bb  "
assert fmt_row([100,"Value"], [5, 10]) == "100   | Value     " # Test with number and longer text

### Challenge

10) Slugify
- slugify(title) → lowercase, trim, replace runs of non-alnum with single -, remove leading/trailing -.
- def slugify(title):
-     ...
- assert slugify("Hello,  World!!") == "hello-world"

In [97]:
import re

def slugify(title):
    """
    Converts a string title into a URL-friendly slug using regular expressions.
    The process includes lowercasing, replacing non-alphanumeric runs with '-', 
    and trimming extraneous hyphens.
    """
    
    # 1. Convert to lowercase
    s = title.lower()
    
    # 2. Replace all runs of non-alphanumeric characters (and whitespace) with a single hyphen.
    #    - The pattern r'[^a-z0-9]+' matches one or more characters that are NOT 
    #      a lowercase letter (a-z) or a digit (0-9).
    #    - This single step handles replacing spaces, commas, exclamation marks, 
    #      and multiple hyphens with one hyphen.
    s = re.sub(r'[^a-z0-9]+', '-', s)

    # 3. Remove any leading or trailing hyphens.
    #    This catches cases where the original string started or ended with a 
    #    non-alphanumeric character (e.g., " Hello " -> "-hello-" -> "hello").
    s = s.strip('-')

    return s

# Test Assertions
assert slugify("Hello,  World!!") == "hello-world"
assert slugify("  Python 3.10 is great! ") == "python-3-10-is-great" # Handles leading/trailing spaces and dots/spaces
assert slugify("What's-Up-Doc?") == "what-s-up-doc" # Handles mixed punctuation and hyphens
print(slugify("What's-Up-Doc?"))

what-s-up-doc


In [99]:
def slugify_basic(title):
    """
    Converts a string title into a URL-friendly slug using basic string operations 
    and iteration, avoiding the 're' module.
    """
    
    # 1. Convert to lowercase
    s = title.lower()
    
    temp_slug = []
    
    # 2. Iterate and replace non-alphanumeric characters with a hyphen
    for char in s:
        # Check if the character is alphanumeric or a digit
        if char.isalnum():
            temp_slug.append(char)
        else:
            # If it is NOT alphanumeric, replace it with a hyphen.
            temp_slug.append('-')
            
    # The string now might look like: "hello---world--"
    s = "".join(temp_slug)
    
    # 3. Collapse multiple hyphens into a single hyphen
    # We use a simple while-loop to replace all occurrences of '--' with '-' until none are left.
    while '--' in s:
        s = s.replace('--', '-')
        
    # 4. Remove any leading or trailing hyphens created during the process
    s = s.strip('-')

    return s

# Test Assertions
assert slugify_basic("Hello,  World!!") == "hello-world"
assert slugify_basic("  Python 3.10 is great! ") == "python-3-10-is-great"
assert slugify_basic("What's-Up-Doc?") == "what-s-up-doc"